In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# 1. Verification Gate
registry_file = "location_registry.csv"
if not os.path.exists(registry_file):
    raise FileNotFoundError("Missing baseline registry! Please run Notebook 01A first.")

In [3]:
# Load primary location coordinates and categories
df_registry = pd.read_csv(registry_file)

In [4]:
# Keep only the core identity metrics for our transport asset
df_transport = df_registry[['location_id', 'location', 'destination_type', 'elevation']].copy()

In [5]:
print("--- NOTEBOOK 04 INGESTION LAYER ---")
print(f"Loaded {len(df_transport)} unique destinations for infrastructural mapping.")
df_transport.head(3)

--- NOTEBOOK 04 INGESTION LAYER ---
Loaded 10 unique destinations for infrastructural mapping.


,location_id,location,destination_type,elevation
0,LOC001,Manali,Mountain Hill Station,1864.0
1,LOC002,Shimla,Mountain Hill Station,2195.0
2,LOC003,Mussoorie,Mountain Hill Station,2006.0


## Topological Complexity with Elevation Variance Modifiers

In [6]:
# 1. Base Transport Complexity Map
complexity_map = {
    "Coastal Plain": 20, "Semi-Arid Plain": 25, 
    "Mountain Hill Station": 70, "High Altitude Desert": 90
}

# UPGRADE: Inject fine-grained variance using elevation as a continuous modifier
df_transport['transport_complexity_score'] = (
    df_transport['destination_type'].map(complexity_map) + (df_transport['elevation'] / 500.0)
).clip(upper=100.0).round(2)

In [7]:
# 2. Elevation Penalty (Logistical and emergency response lag calculation)
df_transport['elevation_penalty'] = (df_transport['elevation'] / 100.0).clip(upper=50.0).round(2)

In [8]:
# 3. Map Road Accessibility Score (Higher = easier physical infrastructure network)
accessibility_map = {
    "Coastal Plain": 95, "Semi-Arid Plain": 90, 
    "Mountain Hill Station": 60, "High Altitude Desert": 40
}
df_transport['road_accessibility_score'] = df_transport['destination_type'].map(accessibility_map)

In [9]:
#4. Map Emergency Access Score (Ease of evacuation under disaster circumstances)
emergency_map = {
    "Coastal Plain": 90, "Semi-Arid Plain": 85, 
    "Mountain Hill Station": 55, "High Altitude Desert": 35
}
df_transport['emergency_access_score'] = df_transport['destination_type'].map(emergency_map)

print("Topological complexity with elevation modifiers and access scores engineered successfully.")
df_transport[['location', 'destination_type', 'elevation', 'transport_complexity_score']].head(5)

Topological complexity with elevation modifiers and access scores engineered successfully.


,location,destination_type,elevation,transport_complexity_score
0,Manali,Mountain Hill Station,1864.0,73.73
1,Shimla,Mountain Hill Station,2195.0,74.39
2,Mussoorie,Mountain Hill Station,2006.0,74.01
3,Nainital,Mountain Hill Station,1938.0,73.88
4,Leh,High Altitude Desert,3414.0,96.83


## Economic Cost Indices & Multi-Factor Budget Stress

In [10]:
# 1. Map Travel Cost Index (Inherent travel expenses due to geographic isolation)
cost_map = {
    "Semi-Arid Plain": 30, "Coastal Plain": 40, 
    "Mountain Hill Station": 70, "High Altitude Desert": 95
}
df_transport['travel_cost_index'] = df_transport['destination_type'].map(cost_map)

In [11]:
# 2. UPGRADE: Compute Multi-Factor Budget Stress Index Formula
df_transport['budget_stress_index'] = (
    (df_transport['travel_cost_index'] * 0.6) + 
    (df_transport['transport_complexity_score'] * 0.3) +
    (df_transport['elevation_penalty'] * 0.1)
)

# Convert to a clean, rounded integer representation
df_transport['budget_stress_index'] = df_transport['budget_stress_index'].round().astype(int)

In [12]:
# 3. Export the finished Dimension Table straight-to-disk
output_features_csv = "transport_features.csv"
df_transport.to_csv(output_features_csv, index=False)

print("=====================================================================")
print(f"🎉 TASK 04 METRIC ENGINE COMPLETE: Saved to '{output_features_csv}'")
print("=====================================================================\n")

# Preview final structural outputs with full variance details
df_transport[['location', 'transport_complexity_score', 'elevation_penalty', 'travel_cost_index', 'budget_stress_index']]

🎉 TASK 04 METRIC ENGINE COMPLETE: Saved to 'transport_features.csv'



,location,transport_complexity_score,elevation_penalty,travel_cost_index,budget_stress_index
0,Manali,73.73,18.64,70,66
1,Shimla,74.39,21.95,70,67
2,Mussoorie,74.01,20.06,70,66
3,Nainital,73.88,19.38,70,66
4,Leh,96.83,34.14,95,89
5,Darjeeling,74.01,20.03,70,66
6,Goa,20.20,0.98,40,30
7,Jaipur,25.86,4.31,30,26
8,Munnar,72.92,14.61,70,65
9,Ooty,74.47,22.35,70,67


## Production Architecture Sanity Verification

In [13]:
print("=====================================================================")
print("📊 PORTFOLIO SANITY CHECK: GRANULAR SCORE DISTRIBUTION")
print("=====================================================================")
# Display the entire table to verify variance distribution
print(df_transport[['location', 'destination_type', 'transport_complexity_score', 'elevation_penalty', 'budget_stress_index']])

📊 PORTFOLIO SANITY CHECK: GRANULAR SCORE DISTRIBUTION
     location       destination_type  transport_complexity_score  \
0      Manali  Mountain Hill Station                       73.73   
1      Shimla  Mountain Hill Station                       74.39   
2   Mussoorie  Mountain Hill Station                       74.01   
3    Nainital  Mountain Hill Station                       73.88   
4         Leh   High Altitude Desert                       96.83   
5  Darjeeling  Mountain Hill Station                       74.01   
6         Goa          Coastal Plain                       20.20   
7      Jaipur        Semi-Arid Plain                       25.86   
8      Munnar  Mountain Hill Station                       72.92   
9        Ooty  Mountain Hill Station                       74.47   

   elevation_penalty  budget_stress_index  
0              18.64                   66  
1              21.95                   67  
2              20.06                   66  
3              19.38 

In [14]:
print("\n=====================================================================")
print("🔍 RELATIVE COMPLEXITY GRADIENT VERIFICATION")
print("=====================================================================")
leh_score = df_transport[df_transport['location'] == 'Leh']['transport_complexity_score'].values[0]
manali_score = df_transport[df_transport['location'] == 'Manali']['transport_complexity_score'].values[0]
goa_score = df_transport[df_transport['location'] == 'Goa']['transport_complexity_score'].values[0]


🔍 RELATIVE COMPLEXITY GRADIENT VERIFICATION


In [15]:
print(f"Leh Complexity ({leh_score}) > Manali Complexity ({manali_score}) > Goa Complexity ({goa_score})")
if leh_score > manali_score > goa_score:
    print("✅ SUCCESS: Continuous variance and gradients are perfectly preserved.")
else:
    print("❌ ERROR: Variance calculation skew detected.")

Leh Complexity (96.83) > Manali Complexity (73.73) > Goa Complexity (20.2)
✅ SUCCESS: Continuous variance and gradients are perfectly preserved.
